# 03. Serving the fine-tuned model with Ray Serve

**AI Drug Discovery & Development Summit 2026 · Hands-on workshop · Part 3 of 3**

**Workshop series:** [01 Ray and offline inference](01_ray_and_offline_inference.ipynb) · [02 Train and tune](02_train_and_tune.ipynb) · [03 Serve](03_serve.ipynb)

Notebook 02 produced a fine-tuned checkpoint. This notebook puts it behind an endpoint a chemist can use:
the 3D results viewer, prediction from Python, and swapping checkpoints without breaking what is already
in use.

| § | Topic | Ray library | Min |
|---|---|---|---|
| 0 | Setup | | 2 |
| 6 | Serving the fine-tuned model | Ray Serve | 17 |
| 7 | Summary and next steps | | 5 |

Section numbers follow the workshop as a whole, so this notebook starts at Section 6.


## 0. Setup

- The Workspace runs the `aiddd-boltz2-of3` image: Ray, Boltz-2 and 2 OpenFold3 versions (0.5 and 0.4.3).
- This notebook runs on a CPU-only head node. L4 GPU nodes join when a cell requests a GPU and leave when idle.
- Data, model weights and all benchmark results live on shared storage.

In [34]:
import os, sys, json, time, warnings, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import ray

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

from aiddd_pipeline import (SHARED, TARGETS, BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT, OF3_OURS_CKPT, OF3_LEGACY,
                            Boltz2Predictor, OF3Predictor, load_ligands, validate, make_scorer)
import workshop_helpers as wh

# Data and weights are downloaded once, ahead of the workshop (README step 2)
needed = [TARGETS[t].path("metadata") for t in wh.TARGET_KEYS] + [BOLTZ_CACHE, OF3_OB_CKPT, OF3_P2_CKPT, OF3_FT_CKPT, OF3_OURS_CKPT]
missing = [str(p) for p in needed if not p.exists()]
assert not missing, f"Missing {missing}. Run `python scripts/setup_data.py` and `python scripts/prepare_mpro.py` first."

# True: run every GPU step live (the presenter, or anyone with their own GPUs).
# False: GPU steps load stored results. Serving is notebook 03.
USE_GPUS = False
SERVICE_URL = os.environ.get("AIDDD_SERVICE_URL", "")   # the deployed prediction service used in notebook 03
GPU = "L4"            # GPU type for every GPU step in this notebook

wh.MINE.mkdir(parents=True, exist_ok=True)
print(f"shared data:  {SHARED}")
print(f"your outputs: {wh.MINE}")

try:
    ray.init(runtime_env={"env_vars": {"AIDDD_SHARED": str(SHARED)}}, ignore_reinit_error=True, logging_level="ERROR")
except ValueError:   # inside an Anyscale Job, which already sets the environment
    ray.init(ignore_reinit_error=True, logging_level="ERROR")
dashboard = os.environ.get("RAY_OVERRIDE_DASHBOARD_URL") or ray.get_runtime_context().dashboard_url
print(f"Ray Dashboard metrics: {dashboard.rstrip('/')}/#/metrics   (open in a new browser tab)")

shared data:  /mnt/user_storage/aiddd
your outputs: /mnt/user_storage/aiddd/workshop/alicia_chua_30e1ffa
Ray Dashboard metrics: https://session-thqdxs9cduqis7hsp7r4mvd42v.i.anyscaleuserdata.com/#/metrics   (open in a new browser tab)


In [ ]:
# Context from notebooks 01 and 02, loaded here so this notebook stands alone
results = {t: wh.load_results(t) for t in wh.TARGET_KEYS}
ev71 = results["ev71-2a"]
x7259a = ev71[ev71.complex_name == "A71EV2A-x7259a"].iloc[0]     # the late follow-on used in 6.3 to 6.5
print("loaded benchmark results:", ", ".join(results))


## 6. Serving with Ray Serve: one base model, many specialists

⏱ **Budget: 17 minutes.**

A chemist does not want a different endpoint per model. Ray Serve **model multiplexing** serves the
general **base model** and every program's **specialist** from one pool of GPU replicas, chosen per
request, and keeps the most recently used checkpoints loaded in memory. This section runs that pool in the Workspace and walks the
lifecycle: start with the base model, predict (it is weak on EV-A71 2A), add the program's specialist,
then swap in a newer version.

### 6.1 Start the pool

`@serve.multiplexed` is the whole mechanism. `__call__` reads the id a request asked for with
`serve.get_multiplexed_model_id()`, and `load` turns that id into a model: here it builds OpenFold3 and
reads the checkpoint, about 5 to 10 s, once. Ray caches what `load` returns per id, so later requests for
that id skip it, and routes each id to a replica that already holds it. Past
`max_num_models_per_replica` (2 here), a new id evicts the least recently used one, and Ray calls its
`__del__`, which frees it.

The registry (`user_config`) lists the base model and both specialists. Nothing loads until a request
asks for an id. In production, adding an id to a running pool is a config change; the Service at the end
of this section does it in place, with no restart.

The replica runs in a different Python from this notebook: OF3-p2 checkpoints only load in OpenFold3
0.4.x, installed separately in the image. `runtime_env={"py_executable": ...}` gives this one deployment
that interpreter; nothing else changes.

In [ ]:
from ray import serve
from aiddd_pipeline import LIVE_MODELS, OF3_LEGACY_PY   # model id -> {version, name, checkpoint}
from of3_live import LiveModel                           # OpenFold3 with its weights loaded, and a pose cache

assert USE_GPUS, "Section 6 starts a GPU pool with serve.run; set USE_GPUS = True (needs an L4)."


@serve.deployment(ray_actor_options={"num_gpus": 0.5, "accelerator_type": "L4",
                                     "runtime_env": {"py_executable": OF3_LEGACY_PY}})   # OpenFold3 0.4.x
class PoseModel:
    def reconfigure(self, models):                           # {model_id: {version, name, checkpoint}}
        self.models = models

    @serve.multiplexed(max_num_models_per_replica=2)         # 2 models in memory; a third evicts the least used
    async def load(self, model_id):
        return LiveModel(self.models[model_id])              # builds OpenFold3 and reads the checkpoint, once

    async def __call__(self, smiles, fresh=False):
        model = await self.load(serve.get_multiplexed_model_id())
        return model(smiles, fresh=fresh)                    # pose, plus where it came from


models = dict(LIVE_MODELS)     # base, ev71-v1, ev71-v2: registered now, each loaded on its first request
handle = serve.run(PoseModel.options(user_config=models).bind(), name="aiddd-pose")

### 6.2 Compounds for the demo

All are EV-A71 2A late follow-ons with a crystal structure, so you can compare the prediction with the
experiment. None was in either fine-tune's training set (the fragment screen).

| Compound | Base model | Our fine-tune (v1) | Public fine-tune (v2) | SMILES |
|---|---|---|---|---|
| **x7259a**, the demo: most potent here (pKD 7.9) | 5.3 Å | 1.0 Å | 0.4 Å | `CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7608a** | 4.8 Å | 0.3 Å | 0.5 Å | `CSc1ncnc(N(C)CC(F)(F)F)c1CC(=O)N1C[C@@H](C(N)=O)C[C@@H]2C[C@@H]21` |
| **x7024a** | 6.1 Å | 1.1 Å | 0.8 Å | `CSc1ncnc(N2CCC3(CCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x6714a** | 4.7 Å | 0.9 Å | 0.4 Å | `NC(=O)[C@H]1CCC(=O)N(c2cc3cnccc3c(F)c2F)C1` |
| **x7011a** | 4.8 Å | 1.4 Å | 0.5 Å | `CC[C@H]1CCCN1c1ncnc(SC)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7101a**, harder: only v2 gets it | 6.2 Å | 4.3 Å | 0.8 Å | `CSc1ncnc(N2CCC3(CCCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |

Ligand RMSD to the crystal; under 2 Å is a correct pose. One diffusion sample, seed 42, as this
notebook runs them.

### 6.3 Predict with the base model, live

`handle.options(multiplexed_model_id="base")` is the one line that picks a model per request. The base
model has never seen this pocket with a ligand in it, so on an EV-A71 2A follow-on it is usually wrong.

Poses are cached per model version, and the presenter runs every step once before the session, so 6.4
and 6.5 come back at once. This step runs live on purpose (`FORCE_COLD`): its cached pose is deleted
first, and you see the real cost, loading the model included. Each answer says where it came from.

In [ ]:
FORCE_COLD = ["base"]   # model ids whose cached pose is deleted first, so the model really runs
COLD = {"base": 40, "ev71-v1": 35, "ev71-v2": 40}   # live on one L4, model load included (s)


async def pose(model_id, smiles, label):
    """One pose from the pool, and where it came from: the pose cache, a model already in
    memory, or a model loaded for this request."""
    out = await handle.options(multiplexed_model_id=model_id).remote(smiles, fresh=model_id in FORCE_COLD)
    load = f", {out['load_seconds']:.0f} s of it loading the model" if out["load_seconds"] else ""
    if out["source"] == "cache":
        print(f"{label}: pose from cache in {out['seconds']:.1f} s{load} (live on one L4: about {COLD[model_id]} s)")
    else:
        where = "a model already in memory" if out["source"] == "memory" else "the model loaded for this request"
        print(f"{label}: computed live in {out['seconds']:.0f} s by {where}{load}")
    return out["cif_path"]


SMILES = "CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1"     # x7259a
crystal = x7259a

cif = await pose("base", SMILES, "x7259a, base model")
wh.show_analog(cif, crystal, label="x7259a (base model)", height=340)

### 6.4 The program's specialist

Notebook 02 fine-tuned the base model on this program's fragment screen: model id `ev71-v1`. It loads on
the first request for its id, next to the base model, which stays loaded and keeps serving everyone else.

In [ ]:
cif = await pose("ev71-v1", SMILES, "x7259a, specialist v1")     # our fine-tune from notebook 02
wh.show_analog(cif, crystal, label="x7259a (specialist v1)", height=340)

Same molecule, same endpoint, a different `multiplexed_model_id`: the pose moves from 5.3 Å off the
crystal to 1.0 Å. One replica now holds both models.

### 6.5 A newer version

A stronger fine-tune arrives (here, the OpenBind release: 0.4 Å on x7259a, and it also gets x7101a), under
its own id, `ev71-v2`. v1 stays reachable by id, for a rollback or a canary. The replica keeps 2 models in
memory, so loading v2 evicts the least recently used one: the base model.

In [ ]:
cif = await pose("ev71-v2", SMILES, "x7259a, specialist v2")     # the OpenBind release
wh.show_analog(cif, crystal, label="x7259a (specialist v2)", height=340)

The eviction is visible: the base model was evicted to make room for v2, so the next base request loads
it again (and evicts v1, now the least recently used). Raise `max_num_models_per_replica`, or add
replicas, when more models are in steady use.

In [ ]:
cif = await pose("base", SMILES, "x7259a, base model again")

**The same pool as an Anyscale Service.** The workshop Service (`jobs/service.yaml`, app `serve_viewer.py`)
runs this pattern behind the web viewer. `OF3Pool` is a multiplexed deployment like `PoseModel`: its
registry is its `user_config`, and the Viewer's `route` names the model id that answers EV-A71 2A requests.
Each demo step adds a model id and repoints the route. `--in-place` applies it with `reconfigure()`: no
restart, no new nodes, and the models already in memory stay there. The viewer shows where each answer
came from, which models the replica holds, and each version's answer for the same molecule.

```bash
anyscale service deploy -f jobs/service.yaml                     # once, ahead of time (5 to 10 min);
                                                                 # add --in-place if it is already running
anyscale service deploy -f jobs/demo_1_base.yaml --in-place       # base model only
anyscale service deploy -f jobs/demo_2_ft.yaml --in-place         # add our fine-tune, v1
anyscale service deploy -f jobs/demo_3_next_ft.yaml --in-place    # swap in v2
```

Edit no file in the repo between the first deploy and the steps: the upload would then carry a code
change, and each step would restart every replica instead of reconfiguring it. Notebooks are not
uploaded, so running and editing this one is safe.

A change to code or the image deploys without `--in-place` and can shift traffic gradually:
`--canary-percent 10`, then `--canary-percent 100` to promote, or `anyscale service rollback -n aiddd-viewer`.

| What Ray Serve and Anyscale add | Why it matters |
|---|---|
| **Model multiplexing** | The base model and every specialist on the same GPUs, chosen per request |
| **In-place updates** | A new model id through `user_config` and `reconfigure()`: no restart, no new nodes |
| **GPUs from every zone** | When one zone runs out of L4s, replicas start in another |
| **Canary rollouts and rollback** | A new model reaches a share of users first |

## 7. Summary and next steps

**Findings**

- **Generalization.** Models that reach 88 to 92% on SARS-CoV-2 late follow-ons fall to 1 to 24% on EV-A71 2A.
- **Fine-tuning.** 66 fragment-screen structures measurably improve OpenFold3 on held-out follow-on
  compounds (notebook 02), and the result is a specialist for that target.

**Delivered on a single Ray cluster**

- **Ray Data.** The full benchmark across 3 campaigns, and 402 new OpenFold3 predictions in 19 minutes on
  12 L4 GPUs.
- **Ray Train.** OpenFold3's own training code on 8 GPUs, with checkpoints and automatic recovery.
- **Ray Serve.** The base model and program specialists in one multiplexed deployment: chosen per
  request, the most recently used kept in memory. On an Anyscale Service, new versions are added in
  place with no restart, and canary rollouts are one flag.

The [README](README.md) has the scientific background and the full runbook. Run the next cell to stop the
pool and release the GPU nodes.

In [ ]:
serve.shutdown()
print("Pool stopped. Idle GPU nodes will be released within a few minutes.")